# Water Quality Data Cleaning: EPA Monitoring Stations

Cleans the EPA/WQX **station metadata** table into a tidy lookup keyed by
`MonitoringLocationIdentifier`. The measurements notebook (`epa-wq-clean.ipynb`)
and the merge step (`src/03_merge/epa-merge-station-and-wq.ipynb`) join against
this file to attach coordinates, HUC, and county to each measurement.

**Input:**  `data/tabular/01_raw/water-quality/epa-stations.csv`
**Output:** `data/tabular/02_clean/water-quality/epa-stations-clean.csv`

**What this notebook does**
1. Loads the raw stations table.
2. Drops columns that are essentially empty (>90% missing).
3. Keeps a curated set of identity / location columns.
4. **Validates coordinates** against an Iowa bounding box and nulls impossible
   values (the raw data contains stray points such as lat `-1.0` / lon `+92.5`).
5. De-duplicates on the station id and writes the result.

> **Path note:** the data tree was migrated to numbered stages
> (`01_raw` / `02_clean` / `03_merged`). This notebook targets that layout.
> The merge notebook and README still reference the old
> `data/tabular/water-quality/{raw,clean}` paths and should be updated to match.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies
    (repo root vs. the notebook folder), so resolving paths relative to a
    fixed number of "../" is fragile. Searching upward for a sentinel makes
    the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "water-quality"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "water-quality"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

## Step 1 — Load

In [ ]:
df_station = pd.read_csv(RAW_DIR / "epa-stations.csv", low_memory=False)
print(f"Loaded {df_station.shape[0]:,} rows x {df_station.shape[1]} columns")
df_station.head()

## Step 2 — Inspect missingness

Understand which columns are worth keeping before pruning.

In [ ]:
missing = (df_station.isna().mean() * 100).round(1).sort_values(ascending=False)
missing.to_frame("percent_missing").head(40)

## Step 3 — Prune and select columns

Treat whitespace-only strings as missing, drop columns that are >90% empty,
then keep a curated set of identity and location fields. `keep_cols` is filtered
to columns that actually survived, so the notebook never errors if the upstream
schema shifts.

In [ ]:
# Whitespace-only cells -> NA (do this on string columns only, not the
# whole frame, so numeric columns are left untouched).
str_cols = df_station.select_dtypes(include="object").columns
df_station[str_cols] = df_station[str_cols].replace(r"^\s*$", pd.NA, regex=True)

# Drop columns that are >90% missing.
keep_threshold = len(df_station) * 0.10
df_station = df_station.dropna(axis=1, thresh=keep_threshold)

keep_cols = [
    "OrganizationIdentifier",
    "MonitoringLocationIdentifier",
    "MonitoringLocationName",
    "MonitoringLocationTypeName",
    "HUCEightDigitCode",
    "LatitudeMeasure",
    "LongitudeMeasure",
    "DrainageAreaMeasure/MeasureValue",
    "DrainageAreaMeasure/MeasureUnitCode",
    "StateCode",
    "CountyCode",
    "ProviderName",
]
keep_cols = [c for c in keep_cols if c in df_station.columns]
df_station = df_station[keep_cols].copy()
print(f"Kept {len(keep_cols)} columns: {keep_cols}")

## Step 4 — Fix data types

In [ ]:
numeric_cols = [
    "LatitudeMeasure", "LongitudeMeasure", "DrainageAreaMeasure/MeasureValue",
]
for col in numeric_cols:
    if col in df_station.columns:
        df_station[col] = pd.to_numeric(df_station[col], errors="coerce")

df_station.dtypes

## Step 5 — Validate coordinates and de-duplicate

The raw measurements file contains physically impossible coordinates, and a
station table should have exactly one row per station id. Iowa lies roughly
within lat `[40.0, 43.7]` and lon `[-96.8, -90.0]`; points outside this box are
data-entry errors, so we null them rather than letting them poison a spatial
join.

In [ ]:
IOWA_BBOX = {"lat": (40.0, 43.7), "lon": (-96.8, -90.0)}

lat, lon = df_station["LatitudeMeasure"], df_station["LongitudeMeasure"]
out_of_box = (
    lat.notna() & ((lat < IOWA_BBOX["lat"][0]) | (lat > IOWA_BBOX["lat"][1]))
) | (
    lon.notna() & ((lon < IOWA_BBOX["lon"][0]) | (lon > IOWA_BBOX["lon"][1]))
)
print(f"Coordinates outside Iowa bounding box (nulled): {int(out_of_box.sum())}")
df_station.loc[out_of_box, ["LatitudeMeasure", "LongitudeMeasure"]] = np.nan

before = len(df_station)
df_station = df_station.drop_duplicates(subset="MonitoringLocationIdentifier")
print(f"Duplicate station ids removed: {before - len(df_station)}")
print(f"Rows with usable coordinates: {df_station['LatitudeMeasure'].notna().sum():,}/{len(df_station):,}")

## Step 6 — Save

In [ ]:
out_file = CLEAN_DIR / "epa-stations-clean.csv"
df_station.to_csv(out_file, index=False)
print(f"Saved {len(df_station):,} stations -> {out_file}")
print("Location types:")
print(df_station["MonitoringLocationTypeName"].value_counts().head(10))